In [2]:
!git config --global user.name "Brendan Hills"
!git config --global user.email brendanhills@google.com

In [3]:
!python3 -m venv venv
!source venv/bin/activate
%pwd


from platform import python_version

print(python_version())

3.11.2


In [4]:
!pip3 -q install db-dtypes
!pip3 -q install "google-cloud-bigquery>=3.17"
!pip3 -q install "google-cloud-aiplatform>=1.38"
!pip3 -q install "pandas>=2.2.0"

In [5]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=WdyeUs2WzBd8v2exSVWBdFuBdyrHI0&access_type=offline&code_challenge=dj-ek0Y3to-OTL2Sb7PhEvzmoMtIf4RUjpSTyAJTYT4&code_challenge_method=S256


Credentials saved to file: [/home/brendanhills/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "uk-bh-experiments-argolis" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [6]:

PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type:"string"}
REGION = "US"  # @param {type: "string"}
DATASET_ID = "schema_mapping"  # @param {type:"string"}

In [7]:
import pandas as pd
from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel
import numpy as np

In [8]:
def get_table_sample(table_name):
  client = bigquery.Client()
  table_query = f"""
  SELECT * FROM {PROJECT_ID}.{DATASET_ID}.{table_name} TABLESAMPLE SYSTEM (10 PERCENT)
  """
  table_sample = client.query(table_query)
  table_sample_df = table_sample.to_dataframe()
  return table_sample_df




In [9]:
TABLENAME1="insurance"

table1_df = get_table_sample(TABLENAME1)

table1_df.head()



,age,sex,bmi,children,smoker,region,charges
0,18,female,26.315,0,False,northeast,2198.18985
1,18,female,38.665,2,False,northeast,3393.35635
2,18,female,35.625,0,False,northeast,2211.13075
3,18,female,30.115,0,False,northeast,21344.84670
4,18,male,23.750,0,False,northeast,1705.62450


In [10]:
TABLENAME2="df1_loan"

table2_df = get_table_sample(TABLENAME2)

table2_df.head()


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,63,LP001213,Male,True,1,Graduate,False,4945,0.0,NaN,360.0,0.0,Rural,False,4945.0
1,127,LP001449,Male,False,0,Graduate,False,3865,1640.0,NaN,360.0,1.0,Rural,True,5505.0
2,284,LP001922,Male,True,0,Graduate,False,20667,0.0,NaN,360.0,1.0,Rural,False,20667.0
3,322,LP002054,Male,True,2,Not Graduate,False,3601,1590.0,NaN,360.0,1.0,Rural,True,5191.0
4,231,LP001768,Male,True,0,Graduate,<NA>,3716,0.0,42.0,180.0,1.0,Rural,True,3716.0


In [11]:
def get_embedding_for_col(column):
  embeddings = []
  #print(f'{column=}')
  for row in column:
    embeddings.append(row)
  return embeddings

def get_embeddings_for_table(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col(table[col]))
  return embeddings


def text_embedding(text):
    """Text embedding with a Large Language Model."""
    model = TextEmbeddingModel.from_pretrained("textembedding-gecko")
    embeddings = model.get_embeddings(text)
    embedding_vector = []
    for embedding in embeddings:
        embedding_vector.append(embedding.values)
    return embedding_vector

In [12]:
def get_embedding_for_col2(column):
  print(f'{column=}')
  #convert column to list of strings
  col_strings = [str(cell) for cell in column]
  #convert list of strings to string
  col_string = ' '.join(col_strings)
  print(f'{col_string=}')
  embeddings = text_embedding([col_string])
  return embeddings

def get_embeddings_for_table2(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col2(table[col]))
  return embeddings




In [13]:
#embeddings_df1 = pd.DataFrame(get_embeddings_for_table2(table1_df))

#embeddings_df1.head()

In [14]:
## return a dataframe of embeddings - one column for each column of the table
def get_embeddings_for_table_columns(table):
    columns_strings = []
    #convert each column of the table into a list of strings
    for col in table.columns:
        print(f'{col=}')
        #convert column to list of strings
        col_strings = [str(cell) for cell in table[col]]
        #convert list of strings to string
        col_string =  col + ' ' + ' '.join(col_strings)
        columns_strings.append(col_string)


    column_embeddings = text_embedding(columns_strings)
    embeddings_df = pd.DataFrame( index=table.columns, data=column_embeddings).transpose()
    return embeddings_df
    

In [15]:
embeddings_df1 = get_embeddings_for_table_columns(table1_df)

print(f'{embeddings_df1=}')

col='age'
col='sex'
col='bmi'
col='children'
col='smoker'
col='region'
col='charges'
embeddings_df1=          age       sex       bmi  children    smoker    region   charges
0   -0.005758 -0.026280 -0.019831  0.013002 -0.008276 -0.032914 -0.007755
1   -0.033790 -0.023900 -0.044575  0.006273 -0.051462 -0.009089 -0.014867
2   -0.031053 -0.048672 -0.047160 -0.063972 -0.056350 -0.059915 -0.047207
3    0.016368  0.034613  0.012239  0.022219 -0.033358 -0.007890 -0.027901
4    0.066797  0.076315  0.081769  0.056994  0.090243  0.052379  0.058458
..        ...       ...       ...       ...       ...       ...       ...
763  0.027498  0.012564  0.016456  0.007373  0.017253 -0.010496  0.009625
764  0.013794  0.034391  0.032202  0.038023  0.044376  0.003344  0.037005
765  0.035183  0.029095  0.071449  0.021733  0.047151  0.021966  0.009552
766 -0.035951 -0.031735 -0.042045 -0.041867 -0.004572 -0.026187 -0.063450
767  0.040707  0.051457  0.020610  0.043942  0.034555  0.035172  0.029288

[768 rows x

In [16]:
embeddings_df2 = get_embeddings_for_table_columns(table2_df)

print(f'{embeddings_df2=}')

col='int64_field_0'
col='Loan_ID'
col='Gender'
col='Married'
col='Dependents'
col='Education'
col='Self_Employed'
col='ApplicantIncome'
col='CoapplicantIncome'
col='LoanAmount'
col='Loan_Amount_Term'
col='Credit_History'
col='Property_Area'
col='Loan_Status'
col='Total_Income'
embeddings_df2=     int64_field_0   Loan_ID    Gender   Married  Dependents  Education  \
0        -0.009048 -0.003375 -0.054041  0.007624    0.000466  -0.013971   
1        -0.005334 -0.038232 -0.040774 -0.041861   -0.013147  -0.069268   
2        -0.067017 -0.037490 -0.049134 -0.043811   -0.040650  -0.025301   
3         0.002941  0.012747  0.029329  0.013934    0.006707   0.016077   
4         0.056794  0.039108  0.096589  0.056364    0.096474   0.101011   
..             ...       ...       ...       ...         ...        ...   
763       0.040192 -0.011658  0.022747  0.002161    0.016805   0.015095   
764       0.027466  0.014721  0.030719  0.018220    0.044630   0.022037   
765       0.031974  0.071224  0.

In [17]:
def vector_similarity(vec1, vec2):
    return np.dot(np.squeeze(np.array(vec1)),np.squeeze(np.array(vec2)))

In [19]:

#print (f'{distances_df=}')
rows_list = []
for row in embeddings_df1:
    row_distances = []
    for col in embeddings_df2:
        distance = vector_similarity(embeddings_df1[row], embeddings_df2[col])
        #print(f'{distance=}')
        row_distances.append(distance)    
    rows_list.append(row_distances)
        

distances_df = pd.DataFrame(index=embeddings_df1.columns,columns=embeddings_df2.columns, data=rows_list)
print (f'{distances_df=}')

#find the smallest cell
smallest =  distances_df.min(axis=None)

print(f'{smallest=}')             


distances_df=          int64_field_0   Loan_ID    Gender   Married  Dependents  Education  \
age            0.791707  0.752721  0.790705  0.730175    0.703998   0.783916   
sex            0.672623  0.648375  0.922214  0.731498    0.664337   0.757912   
bmi            0.753886  0.684646  0.720318  0.688530    0.644809   0.713240   
children       0.723871  0.713788  0.693772  0.657793    0.809987   0.659830   
smoker         0.630136  0.626761  0.731927  0.801056    0.666913   0.714262   
region         0.635808  0.611648  0.684321  0.624576    0.598867   0.708078   
charges        0.773075  0.749131  0.650246  0.639351    0.677402   0.670167   

          Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  \
age            0.704201         0.770262           0.743515    0.756378   
sex            0.692428         0.660796           0.668510    0.686383   
bmi            0.653830         0.770702           0.733357    0.760406   
children       0.639355         0.653653      